# Project 4: Real-Time Event Streaming Analytics with Watermarking and Checkpoints

**Module**: 13_end_to_end_projects  
**Notebook**: 04_streaming_analytics_pipeline.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Complete end-to-end real-time streaming pipeline: simulating incoming JSON event streams, tumbling window aggregations, watermarking, checkpointing, and failure recovery.
- **Why does this topic exist?**: Real-time stream processing is a cornerstone of modern event-driven architectures.
- **What problem does it solve?**: Handling streaming event ingestion without external Kafka dependencies in local Docker clusters.
- **Where is it used in real-world Data Engineering?**: Live operational telemetry and real-time revenue monitoring dashboards.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Advanced |
| **Estimated effort** | 4–6 h |
| **Prerequisites** | Structured Streaming, watermarks, checkpoints, sinks |
| **Required depth** | Deep working knowledge: can build a recoverable streaming analytics pipeline |
| **Data Engineer** | Implement an event-time streaming pipeline with watermarks and durable output. |
| **Senior Data Engineer** | Design state, checkpoint, and sink semantics for production. |
| **Spark specialist** | Reason about late data, delivery guarantees, and operational monitoring. |
| **Next topic** | `../README.md` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

A streaming analytics pipeline combines ingestion, event-time processing, stateful aggregation, and durable
output. The hard parts are not the aggregation syntax; they are deciding how late data is handled, how state is
bounded, how recovery avoids duplicates, and how the pipeline is monitored. This project integrates the streaming
module into an end-to-end design.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Event time** | When the event happened | Drives windows and watermarks |
| **Watermark** | Lateness bound | Bounds state and defines late data |
| **State store** | Aggregation state | Recovered from the checkpoint |
| **Checkpoint** | Progress and state | Offsets, commits, metadata |
| **Sink semantics** | Delivery guarantee | Idempotent or transactional output |
| **Monitoring** | Operational health | Input rate, batch duration, state size, late data |

### How it works, step by step

1. Ingest events from a replayable source.
2. Parse and validate the event schema.
3. Apply event-time windows and a watermark.
4. Aggregate statefully and update the result.
5. Write to a sink with idempotent or transactional semantics.
6. Commit offsets and state to the checkpoint.
7. Monitor latency, state, and late-data rates.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import os
import time

spark = (
    SparkSession.builder
    .appName("Streaming_Analytics_Project")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Setting Up Directory Stream Paths and Synthetic Event Generator

In [ ]:
import json

stream_input = "/opt/spark-data/streaming/events"
chk_path = "/opt/spark-data/warehouse/project_stream_chk"
out_path = "/opt/spark-data/warehouse/project_stream_out"

# Reset directories
os.system(f"rm -rf {stream_input}/* {chk_path} {out_path}")
os.makedirs(stream_input, exist_ok=True)

event_schema = StructType([
    StructField("order_id", IntegerType()),
    StructField("order_ts", StringType()),
    StructField("channel", StringType()),
    StructField("revenue", DoubleType())
])

# Read streaming files
events_stream = (
    spark.readStream
    .schema(event_schema)
    .json(stream_input)
    .withColumn("timestamp", F.to_timestamp("order_ts"))
    .withWatermark("timestamp", "10 seconds")
)

print("Streaming input table configured. isStreaming:", events_stream.isStreaming)

## 4. Real-Time Windowed Revenue Aggregations

In [ ]:
# Windowed revenue by channel over 5-second tumbling windows
windowed_rev = (
    events_stream
    .groupBy(
        F.window("timestamp", "5 seconds"),
        F.col("channel")
    )
    .agg(
        F.count("order_id").alias("order_count"),
        F.round(F.sum("revenue"), 2).alias("total_channel_revenue")
    )
)

# Start streaming query writing to memory sink
query = (
    windowed_rev
    .writeStream
    .format("memory")
    .queryName("live_sales_dashboard")
    .outputMode("complete")
    .start()
)

print(f"Streaming query started with Query ID: {query.id}")

---

## Experiment: Simulating Micro-Batch Event Ingestion and Live Querying

- **1. What are we testing?**: Drop event files into stream_input and query the live_sales_dashboard table.
- **2. What do we expect?**: Dashboard updates in real time as files are written.

In [ ]:
# Simulate 2 event bursts
for batch_idx in range(1, 3):
    events = [
        {"order_id": 100 + batch_idx*10 + i, "order_ts": "2026-11-09 20:00:00", "channel": "Mobile" if i%2==0 else "Web", "revenue": 100.0 * i}
        for i in range(1, 4)
    ]
    with open(f"{stream_input}/batch_{batch_idx}.json", "w") as f:
        for ev in events:
            f.write(json.dumps(ev) + "\n")
    time.sleep(2)

query.processAllAvailable()

print("=== LIVE STREAMING SALES DASHBOARD ===")
spark.sql("SELECT window.start, window.end, channel, order_count, total_channel_revenue FROM live_sales_dashboard ORDER BY total_channel_revenue DESC").show(truncate=False)

query.stop()
print("Streaming pipeline completed and stopped.")

**3. What happened?**: Live streaming aggregation updated dynamically as new files landed.  
**4. Why did it happen?**: Micro-batch query recomputed the windowed result table in Complete mode.  
**5. Practical Engineering Lesson**: Structured Streaming provides unified DataFrame syntax for both real-time event streams and batch files.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Clickstream analytics

- **Industry**: Digital media
- **Scenario**: A media company needs minute-level page views and session metrics.
- **Problem**: Events arrive out of order and some are delayed by mobile retries.
- **Why this concept is relevant**: Event-time windows with a watermark produce stable metrics and bounded state.
- **Implementation idea**: Read from Kafka, watermark by event time, aggregate per window, and write to an upsert store.
- **Expected result**: Fresh, recoverable metrics with controlled late-data loss.
- **Alternative**: Batch micro-runs every few minutes.
- **Trade-offs**: Micro-batch is simpler but has higher latency and less sophisticated state handling.

### Use Case 2: Fraud alert pipeline

- **Industry**: Financial services
- **Scenario**: Transactions must be scored and alerted within seconds.
- **Problem**: A missed or duplicated alert can have financial and regulatory consequences.
- **Why this concept is relevant**: A replayable source, checkpointed state, and idempotent alert sink provide recoverable processing.
- **Implementation idea**: Stream transactions, join risk features, write alerts with a deterministic alert ID.
- **Expected result**: Timely alerts with no duplicates after recovery.
- **Alternative**: Score in the application before publishing.
- **Trade-offs**: Edge scoring reduces latency but duplicates logic and complicates model updates.

### Use Case 3: IoT fleet monitoring

- **Industry**: Logistics
- **Scenario**: Vehicle telemetry arrives continuously and drives maintenance alerts.
- **Problem**: Sensors buffer data during connectivity gaps, causing late arrivals.
- **Why this concept is relevant**: A watermark based on the observed delay bounds state while accepting the agreed lateness.
- **Implementation idea**: Aggregate per vehicle and window, then write to a time-series store.
- **Expected result**: Stable monitoring with predictable state size.
- **Alternative**: Store raw events and compute alerts in batch.
- **Trade-offs**: Batch computation is simpler but delays maintenance alerts.

---

## Business Value

- Timely operational insight.
- Recoverable processing with bounded state.
- Clear late-data and delivery semantics.
- Lower operational risk through monitoring.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Use a replayable source and a unique checkpoint.
- Set the watermark from measured lateness.
- Make the sink idempotent or transactional.
- Monitor input rate, batch duration, state size, and late data.
- Test recovery and late-data behavior before production.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Test late and duplicate events

**Steps**

1. Send events with a controlled fraction arriving after the watermark.
2. Restart the query and resend a batch to simulate replay.
3. Verify the output counts and late-data behavior.

**What to observe**: Late events are dropped according to the watermark; replay is safe only with an idempotent sink.

**Healthy result**: Counts match the expected semantics and no duplicates appear.

**Problematic result**: The sink duplicates rows after restart.

**Engineering takeaway**: Streaming correctness is a joint property of watermark, checkpoint, and sink.

---

## Trade-offs and Alternatives

### Advantages

- Low latency
- Event-time correctness
- Recoverable state

### Limitations

- State management
- Late-data trade-offs
- Sink semantics

### When to use it

- Clickstream
- Fraud alerts
- IoT monitoring

### When not to use it

- Unbounded state
- Non-replayable sources without a plan
- Assuming exactly-once without sink support

### Alternatives inside Spark

- Structured Streaming
- Watermarks
- Checkpoints

### Alternatives outside Spark

- Flink
- Kafka Streams
- Managed streaming services

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Check Query Last Progress
**Problem Statement**: Inspect query.lastProgress dictionary.  
**Input / Context**: query

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Parquet Streaming Sink
**Problem Statement**: Write a stream in Append mode to Parquet sink with checkpoint.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Stateful Fault Recovery
**Problem Statement**: Explain how restarting the query with the same checkpoint resumes state.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### State growth after a schema change

**Scenario**: A new event field changes the grouping key and the state store grows rapidly.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check the aggregation key and cardinality.
2. Verify the watermark still applies to the event-time column.
3. Plan a new checkpoint and backfill for the changed query.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Check Query Last Progress):
```python
print('query.lastProgress provides inputRowsPerSecond, processedRowsPerSecond, batchId.')
```

#### Solution 2 (Parquet Streaming Sink):
```python
print('events_stream.writeStream.format("parquet").option("checkpointLocation", chk).start(out)')
```

#### Solution 3 (Stateful Fault Recovery):
```python
print('Structured Streaming loads state store snapshots from state/ and offset WAL from offsets/.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Unified Paradigm**: Batch and Streaming share the exact same DataFrame / SQL APIs.
- **Watermarking & Checkpoints**: Mandatory for state bounding and reliable recovery; end-to-end exactly-once also depends on a replayable source and an idempotent or transactional sink.

### Essential Technical Terms
- `Tumbling Window`: Fixed-duration non-overlapping time window.
- `Complete Output Mode`: Emits full result table on every trigger.

### Common Engineering Mistakes
- **Omitting Checkpoint in Production**: Stream cannot recover from crash without checkpoints.

### Interview Questions on This Topic
- **Q: What happens if a streaming query crashes and is restarted with the same checkpointLocation?**
  - *A*: It reads the offset log and state store from the checkpoint directory, resuming exactly where it left off with zero data loss or duplicate processing.


### More Interview Questions

- **Q: How do you choose a watermark delay?**
  - *A*: From the measured distribution of event lateness and the business tolerance for dropping late events, balanced against state size.
- **Q: What makes a streaming sink safe for replay?**
  - *A*: Idempotent writes keyed by a deterministic ID or transactional commits that align with checkpoint progress.

### What to Study Next
Curriculum complete! Review documentation in docs/ and project README.md.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')